# Breast Cancer (BCSC) - Pré-processamento

Transforma o `breast_cancer_screening_bcsc.csv` bruto em uma tabela 100% numérica e sem valores ausentes, pronta para os modelos.
Os motivos de cada passo estão em [DICIONARIO_DE_DADOS.md](../dataset/DICIONARIO_DE_DADOS.md).

Variável alvo: `Is_Binary_Indicator_Of_Cancer_Diagnosis` (1 = câncer diagnosticado em até 1 ano após a mamografia).

In [3]:
import pandas as pd

df = pd.read_csv("../dataset/breast_cancer_screening_bcsc.csv")
print(df.shape)

(39998, 13)


## 1. Remoção de colunas

- `Cancer_Type`: vazamento de dados. É só o alvo com mais detalhes.
- `Body_Mass_Index`: 58% de valores ausentes. Preencher mais da metade da coluna seria inventar dados.

`Patients_Study_ID` **fica** no arquivo: não é uma feature, mas é necessário na modelagem para manter
todos os exames de uma mesma mulher do mesmo lado na divisão treino/teste.

In [4]:
df = df.drop(columns=["Cancer_Type", "Body_Mass_Index"])

## 2. Duplicados

Não há linhas repetidas. Exames com as mesmas características, mas de mulheres diferentes, são legítimos e ficam.

In [5]:
print(f"Linhas duplicadas: {df.duplicated().sum()}")

Linhas duplicadas: 0


## 3. Valores ausentes

As colunas Yes/No têm entre 0,6% e 11,7% de ausentes. Preenchemos com o valor mais frequente (a moda) de cada coluna.

In [6]:
colunas_sim_nao = [
    "Comparison_Mammogram_From_Mammography",
    "Family_History_Of_Breast_Cancer",
    "Current_Use_Of_Hormone_Therapy",
    "Binary_Indicator",
    "History_Of_Breast_Biopsy",
]

print(df[colunas_sim_nao].isna().sum())

for coluna in colunas_sim_nao:
    df[coluna] = df[coluna].fillna(df[coluna].mode()[0])

Comparison_Mammogram_From_Mammography    4680
Family_History_Of_Breast_Cancer           228
Current_Use_Of_Hormone_Therapy           1772
Binary_Indicator                          578
History_Of_Breast_Biopsy                  815
dtype: int64


## 4. Codificação

- **Ordinais** (têm ordem): viram números inteiros. A avaliação do radiologista segue a ordem de risco
  observada nos dados (de 0,1% de câncer em "Negative" até 80% em "Highly suggestive of malignancy").
- **Yes/No**: viram 0/1.
- **True/False** (alvo e tipo de mamografia): viram 0/1. Em `Is_Film_Or_Digital_Mammogram`, 1 = digital.

In [7]:
ordinais = {
    "Radiologists_Assessment": {
        "Negative": 1,
        "Benign findings": 2,
        "Probably benign": 3,
        "Needs additional imaging": 4,
        "Suspicious abnormality": 5,
        "Highly suggestive of malignancy": 6,
    },
    "Patients_BI_RADS_Breast_Density": {
        "Almost entirely fatty": 1,
        "Scattered fibroglandular densities": 2,
        "Heterogeneously dense": 3,
        "Extremely dense": 4,
    },
}

for coluna, mapa in ordinais.items():
    df[coluna] = df[coluna].map(mapa)

for coluna in colunas_sim_nao:
    df[coluna] = df[coluna].map({"No": 0, "Yes": 1})

df["Is_Binary_Indicator_Of_Cancer_Diagnosis"] = df["Is_Binary_Indicator_Of_Cancer_Diagnosis"].astype(int)
df["Is_Film_Or_Digital_Mammogram"] = df["Is_Film_Or_Digital_Mammogram"].astype(int)

## 5. Verificação

In [8]:
# Se algum valor não estava no mapa, ele vira NaN
assert df.isna().sum().sum() == 0, "Há valores que não foram mapeados"

print(df.shape)
print(df.dtypes)
print(df["Is_Binary_Indicator_Of_Cancer_Diagnosis"].value_counts(normalize=True).round(4))
df.head()

(39998, 11)
Age_At_The_Time_Of_Mammography             int64
Radiologists_Assessment                    int64
Is_Binary_Indicator_Of_Cancer_Diagnosis    int64
Comparison_Mammogram_From_Mammography      int64
Patients_BI_RADS_Breast_Density            int64
Family_History_Of_Breast_Cancer            int64
Current_Use_Of_Hormone_Therapy             int64
Binary_Indicator                           int64
History_Of_Breast_Biopsy                   int64
Is_Film_Or_Digital_Mammogram               int64
Patients_Study_ID                          int64
dtype: object
Is_Binary_Indicator_Of_Cancer_Diagnosis
0    0.9935
1    0.0065
Name: proportion, dtype: float64


,Age_At_The_Time_Of_Mammography,Radiologists_Assessment,Is_Binary_Indicator_Of_Cancer_Diagnosis,Comparison_Mammogram_From_Mammography,Patients_BI_RADS_Breast_Density,Family_History_Of_Breast_Cancer,Current_Use_Of_Hormone_Therapy,Binary_Indicator,History_Of_Breast_Biopsy,Is_Film_Or_Digital_Mammogram,Patients_Study_ID
0,62,1,0,1,2,0,0,1,0,0,1
1,65,1,0,1,4,0,0,1,0,0,2
2,69,4,0,1,2,0,0,1,0,0,3
3,64,2,0,1,2,0,0,1,0,0,4
4,63,3,0,1,2,0,0,1,1,0,5


## 6. Salvar

A padronização (scaler) e a divisão treino/teste ficam para a etapa de modelagem:
o scaler precisa ser ajustado só com os dados de treino para não vazar informação do teste.

In [9]:
df.to_csv("../dataset/breast_cancer_screening_bcsc_processed.csv", index=False)